In [ ]:
import numpy as np
import pandas as pd
import pickle, time

In [ ]:
PATH = "data/train_triplets.txt"

In [ ]:
user_map, song_map = {}, {}

In [ ]:
def encode(series, mapping):
    for v in series.unique():
        if v not in mapping:
            mapping[v] = len(mapping)
    return series.map(mapping).to_numpy(dtype="int32")

In [ ]:
users, songs, plays = [], [], []
t0 = time.time()

In [ ]:
reader = pd.read_csv(
    PATH, sep="\t", header=None,
    names=["user", "song", "plays"],
    dtype={"plays": "int32"},
    chunksize=2_000_000,
)

In [ ]:
for i, chunk in enumerate(reader):
    users.append(encode(chunk["user"], user_map))
    songs.append(encode(chunk["song"], song_map))
    plays.append(chunk["plays"].to_numpy())
    print(i, len(user_map), len(song_map), f"{time.time()-t0:.0f}s")

In [ ]:
users = np.concatenate(users)
songs = np.concatenate(songs)
plays = np.concatenate(plays)
print(users.shape, songs.shape, plays.shape)

In [ ]:
np.savez("data/triplets_int.npz", user=users, song=songs, plays=plays)
with open("data/id_maps.pkl", "wb") as f:
    pickle.dump((list(user_map), list(song_map)), f)

In [ ]:
d = np.load("data/triplets_int.npz")
users, songs, plays = d["user"], d["song"], d["plays"]
print(users.shape, songs.shape, plays.shape)

In [ ]:
print("min plays:", plays.min())
print("max plays:", plays.max())
print("mean plays:", plays.mean())
print("median plays:", np.median(plays))
print("share of rows with plays == 1:", (plays == 1).mean())

In [ ]:
qs = [0.5, 0.75, 0.9, 0.95, 0.99, 0.999]
print(dict(zip(qs, np.quantile(plays, qs))))

for t in [2, 5, 10, 50, 100, 1000]:
    print(f"plays >= {t}: {(plays >= t).sum():>10,}  ({(plays >= t).mean():.4%})")

In [ ]:
songs_per_user = np.bincount(users)   # number of rows per user -> distinct songs
n_users, n_songs = len(songs_per_user), songs.max() + 1

qs = [0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
print("min/max/mean:", songs_per_user.min(), songs_per_user.max(), songs_per_user.mean())
print(dict(zip(qs, np.quantile(songs_per_user, qs))))

for t in [5, 10, 20, 50]:
    print(f"users with < {t} songs: {(songs_per_user < t).mean():.2%}")
    
print("density:", len(users) / (int(n_users) * int(n_songs)))

In [ ]:
users_per_song = np.bincount(songs)   # number of distinct listeners per song

print("min/max/mean:", users_per_song.min(), users_per_song.max(), users_per_song.mean())
print(dict(zip(qs, np.quantile(users_per_song, qs))))

for t in [2, 5, 10, 50]:
    print(f"songs with < {t} listeners: {(users_per_song < t).mean():.2%}")

top = np.sort(users_per_song)[::-1]
print("share of all rows held by top 100 songs:", top[:100].sum() / len(songs))
print("share of all rows held by top 1% of songs:", top[:len(top)//100].sum() / len(songs))

In [ ]:
for t in [2, 5, 10, 20, 50]:
    keep = users_per_song >= t
    print(f"min {t:>2} listeners: songs kept {keep.sum():>7,} ({keep.mean():.1%}), "
          f"rows kept {keep[songs].mean():.1%}")

In [ ]:
for t in [10, 20, 50]:
    row_mask = (users_per_song >= t)[songs]
    spu = np.bincount(users[row_mask], minlength=int(n_users))
    print(f"min {t:>2} listeners: users with 0 songs {(spu == 0).sum():>6,}, "
          f"< 10 songs {(spu < 10).mean():.1%}, median songs {np.median(spu):.0f}")

In [ ]:
total_plays = np.bincount(users, weights=plays)   # total plays per user

top = np.argsort(plays)[-10:][::-1]               # 10 rows with the highest play counts
for i in top:
    u, s = users[i], songs[i]
    print(f"plays {plays[i]:>5}  song listeners {users_per_song[s]:>6}  "
          f"user songs {songs_per_user[u]:>4}  user total plays {total_plays[u]:>8.0f}")

In [ ]:
n_users, n_songs = int(users.max()) + 1, int(songs.max()) + 1

def filter_iter(min_song, min_user):
    mask = np.ones(len(users), dtype=bool)
    it = 0
    while True:
        song_cnt = np.bincount(songs[mask], minlength=n_songs)
        user_cnt = np.bincount(users[mask], minlength=n_users)
        new = mask & (song_cnt >= min_song)[songs] & (user_cnt >= min_user)[users]
        it += 1
        if new.sum() == mask.sum():
            return new, it
        mask = new

for min_song, min_user in [(20, 10), (50, 10), (20, 20)]:
    m, it = filter_iter(min_song, min_user)
    print(f"songs>={min_song}, users>={min_user}: iters {it}, "
          f"rows {m.sum():,} ({m.mean():.1%}), "
          f"users {len(np.unique(users[m])):,}, songs {len(np.unique(songs[m])):,}")

In [ ]:
mask, it = filter_iter(100, 10)

old_users, new_users = np.unique(users[mask], return_inverse=True)
old_songs, new_songs = np.unique(songs[mask], return_inverse=True)

u = new_users.astype("int32")
s = new_songs.astype("int32")
p = plays[mask]

print("rows:", len(u))
print("users:", len(old_users), " max new user id + 1:", u.max() + 1)
print("songs:", len(old_songs), " max new song id + 1:", s.max() + 1)

In [ ]:
np.savez("data/filtered_100_10.npz",
         user=u, song=s, plays=p,
         old_user=old_users, old_song=old_songs)

## Testing with the filtered data

In [ ]:
d = np.load("data/filtered_100_10.npz")
u, s, p = d["user"], d["song"], d["plays"]

print("rows:", len(u))
print("users:", u.max() + 1, " songs:", s.max() + 1)
print("min songs per user:", np.bincount(u).min())
print("min listeners per song:", np.bincount(s).min())